In [31]:
import anthropic 
import mysql.connector
from dotenv import load_dotenv
import base64
from dateutil import parser

import json
import os
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")
print(anthropic_key)

None


In [27]:
#FUNCTION: Connects to the LLM. Scans the respected pdf. Translates handwritten data into dictionary. Confirms confidence of scan.
client = anthropic.Anthropic()
def read_trf(pdf):
    #Convert PDF into base64
    with open(pdf, "rb") as file:
        pdf_bytes = file.read()
    encoded_bytes = base64.b64encode(pdf_bytes)
    pdf_base64_string = encoded_bytes.decode("utf-8")

    #Retry Loop: call Claude, check confidence, retry up to 3x
    extracted_data = None
    flagged_retry = 0
    while flagged_retry < 3:
        client_message = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=8000,
         messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "document",
                        "source": {
                            "type": "base64",
                            "media_type": "application/pdf",
                            "data": pdf_base64_string
                        }
                    },
                    {
                        "type": "text",
                        "text": (
                            "Extract the handwritten print under each respected field. "
                            "For every field, carefully distinguish between three possible situations. "
                            "First, the field may be genuinely blank, meaning nothing was written, marked, or "
                            "signed by the patient or physician. In this case, set the field's value to the "
                            "exact word 'blank' and give it a high confidence score, since you are certain "
                            "nothing is there. Second, something may be written or marked but it is difficult "
                            "to read clearly, such as messy handwriting or a faint checkbox. In this case, "
                            "provide your best interpretation of the value and give it a low confidence score "
                            "below 0.9, so it can be flagged for review. Third, something may be present but "
                            "visually subtle and easy to overlook, such as a light pencil checkmark or a faint "
                            "signature. Examine every field closely, including checkboxes and signature lines, "
                            "before concluding a field is blank. Do not report a field as 'blank' unless you "
                            "have carefully confirmed there is truly nothing marked there. If you are uncertain "
                            "whether a faint mark exists, treat it as present but unclear, and report it with a "
                            "low confidence score rather than marking it blank."
                            "Ordering Physician Information: Practice/Client Name, Phone, Last Name, First Name, NPI # "
                            "Email, Fax, Street Address, Suite/Bldg #, City, State, Postal Code "
                            "Patient Information: Last Name Required, First Name Required, MI, Date Of Birth(MM/DD/YYYY), Phone, The follow will be a mark of either M or F: Sex at Birth, "
                            "Email(for online report access), Medical Record #, Street Address/PO Box/Building/Floor/etc, City, State/Province/Region, Zip/Postal Code/ Country "
                            "Check which box is checked off, whichever has the majority mark is the one selected, there can only be one. "
                            "Demographic: Race: White, Asian, Black/African American, Native America/ Alaska Native, Native Hawaiian/ Other Pacific Islander, "
                            "Ethnicity: Not Hispanic, Latino, Spanish OR Hispanic, Latino, Spanish "
                            "Patient History: Diabetes AND/OR Family of MI Z82.49, Family History of Heart Attack, Stroke, Coronary Artery Bypass, Stent or Angina, <= "
                            "65 years of age(Parent/sibling/child) AND/OR High Dose Biotin. "
                            "This field will be a checkbox so look for a mark. Billing Information: Client OR Patient Self-Pay, " 
                            "This field will be a checkmark so look for which test is checked. Test Requested: 1003 SmartVascular DX (SVDx) OR 1028 Expanded Lipid Profile(ELP) OR "
                            "1003/1028 Smart Vascular DX and ELP Panel. Specimen Collection: Date Collected in MM/DD/YYYY, Time (AM or PM marked), Collected by(please print) "
                            "Ordering Physician: Just note if signature is there then say: Signature collected, if not say Signature not collected, Date: in MM/DD/YYYY "
                            "This has a checkmark option, but also writing. Patient Acknowledgment: Just note if signature is there then say: Signature collected, if not say Signature not collected, Date: in MM/DD/YYYY. "
                            "Once carefully extracting all fields specified, return all extracted data as a structured JSON object using these extracted field names, and include a "
                            "confidence score between 0 and 1 for each field. Keep all original fields and values in the main structured JSON as specified above. "
                            "Use lowercase snake_case field names as the JSON keys. I will include the exact schema of how i want the json labled. "
                             """Use the following exact field mapping. Do not invent your own key names — match each field to the exact key listed below.

                            Ordering Physician Information:
                            Practice/Client Name -> practice_client_name
                            Phone (in this section) -> ordering_physician_phone
                            Last Name (in this section) -> ordering_physician_last_name
                            First Name (in this section) -> ordering_physician_first_name
                            NPI # -> npi
                            Email (in this section) -> ordering_physician_email
                            Fax -> ordering_physician_fax
                            Street Address (in this section) -> ordering_physician_street_address
                            Suite/Bldg # -> ordering_physician_suite_bldg
                            City (in this section) -> ordering_physician_city
                            State (in this section) -> ordering_physician_state
                            Postal Code -> ordering_physician_postal_code

                            Patient Information:
                            Last Name Required -> patient_last_name
                            First Name Required -> patient_first_name
                            MI -> patient_middle_initial
                            Date of Birth (MM/DD/YYYY) -> date_of_birth
                            Phone (in this section) -> patient_phone
                            Sex at Birth -> sex_at_birth
                            Email (for online report access) -> patient_email
                            Medical Record # -> medical_record_number
                            Street Address/PO Box/Building/Floor/etc -> patient_street_address
                            City (in this section) -> patient_city
                            State/Province/Region -> patient_state
                            Zip/Postal Code -> patient_zip_code
                            Country -> patient_country

                            Demographic:
                            Race (selected option only) -> race
                            Ethnicity (selected option only) -> ethnicity

                            Patient History:
                            Diabetes checkbox -> patient_history_diabetes
                            Family of MI Z82.49 / Family History checkbox -> patient_history_family_heart
                            High Dose Biotin checkbox -> patient_history_high_dose_biotin

                            Billing Information:
                            Client OR Patient Self-Pay (selected option) -> billing_type

                            Test Requested:
                            The single selected test option -> test_requested

                            Specimen Collection:
                            Date Collected -> specimen_collection_date
                            Time (AM or PM) -> specimen_collection_time
                            Collected by -> specimen_collected_by

                            Ordering Physician Signature/Attestation section:
                            Signature presence -> ordering_physician_signature_status (value must be exactly "Signature collected" or "Signature not collected")
                            Date (in this section) -> ordering_physician_date

                            Patient Acknowledgment section:
                            Signature presence -> patient_acknowledgment_signature_status (value must be exactly "Signature collected" or "Signature not collected")
                            Date (in this section) -> patient_acknowledgment_date
                            """
                            "This form contains three separate date fields in different sections that must not be confused "
                            "with one another. Name the date field in the Ordering Physician signature section exactly "
                            "'ordering_physician_date'. Name the date field in the Patient Acknowledgment signature section "
                            "exactly 'patient_acknowledgment_date'. Name the date field in the Specimen Collection section "
                            "exactly 'specimen_collection_date'. Do not use the generic key 'date' for any of these three fields."
                            "Additionally, create a separate JSON object called 'low_confidence_fields' that includes the field name and value for any field where your confidence score is below 0.9."

                        )
                    }
                ]
            }
        ]
    )
        #Claude often wraps its JSON in markdown backticks ('''json''') 
        #Strip them before parsing so json.load() doesnt crash on the markdown backtick
        response_text = client_message.content[0].text
        response_text = response_text.strip()
        response_text = response_text.removeprefix("```json").removeprefix("```")
        response_text = response_text.removesuffix("```")
        response_text = response_text.strip()

        try: 
            extracted_data = json.loads(response_text)
        except json.JSONDecodeError:
            flagged_retry += 1
            continue

        #Only runs if parsing successful
        if not extracted_data["low_confidence_fields"]:
            break

        flagged_retry += 1

    if extracted_data is None:
            return {"status": "failed", "reason": "Could not parse a valid response after 3 attempts"}

    if extracted_data["low_confidence_fields"]:
        extracted_data["permanently_flagged"] = extracted_data["low_confidence_fields"]
    
    return extracted_data



In [28]:
def convert_to_json(extracted_data):
    #Convert all dates to the same MM/DD/YYYY format, if blank returns "blank"
    list_fields_date = ["date_of_birth", "ordering_physician_date", "patient_acknowledgment_date", "specimen_collection_date"]
    for date_types in list_fields_date:
        try:
            parsed_data = parser.parse(extracted_data[date_types])
            structured_date = parsed_data.strftime("%m/%d/%Y")
            extracted_data[date_types] = structured_date
        except (parser.ParserError, ValueError):
            pass
    return extracted_data

In [29]:
result = read_trf("/Users/ianang/downloads/Test Requisition Form.pdf")
print(result)

TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"